In [6]:
from google.colab import files
uploaded = files.upload()


Saving 2023 Active Merchant List (Affirm Inc).xlsx to 2023 Active Merchant List (Affirm Inc).xlsx


In [7]:
!pip install names yfinance pycountry openpyxl tqdm transformers torch us pgeocode pyxlsb

import pandas as pd
import re
from datetime import datetime
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.styles import numbers
from openpyxl.formatting.rule import ColorScaleRule, CellIsRule
from openpyxl.styles import Color
from openpyxl.utils.dataframe import dataframe_to_rows
import names
import pycountry
import yfinance as yf
import time
import string
from requests.exceptions import HTTPError
from tqdm import tqdm
from transformers import pipeline
import us
import pgeocode


In [ ]:
# Register tqdm with pandas
tqdm.pandas()
# Initialize the named entity recognition pipeline
ner = pipeline("token-classification", model="dbmdz/bert-large-cased-finetuned-conll03-english")

# Specify the input file path here
input_file = "2023 Active Merchant List (Affirm Inc).xlsx"
# Specify the sanctioned countries list file path here
sanctioned_countries_file = "Sanctioned Countries List.xlsx"
# Prompt the user for client name and domain input
client_name = input("What is the client name? ")
client_domain = input("What is the client domain? ").lower()

def validate_required_columns(df, required_columns):
    """
    Validates if all required columns are present in the DataFrame.
    :param df: Input DataFrame
    :param required_columns: List of required column names
    :raises ValueError: If any required column is missing
    """
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"The following required columns are missing in the input file: {', '.join(missing_columns)}")

# CREATE A TEMPORARY WORKBOOK TO CAPTURE PRE PROCESSING
def create_addr_gaps_start_workbook(file_path, df):
    # Create a new workbook for "Addr. Gaps to Start"
    wb = Workbook()
    ws = wb.active
    ws.title = "Addr. Gaps to Start"

    # Define headers
    headers = [
        "Total Unique Supplier IDs",
        "Number of Unique Addresses with Gaps",
        "Street Address Gaps",
        "City Gaps",
        "State Gaps",
        "Country Gaps",
        "Zip Code Gaps"
    ]
    ws.append(headers)

    # Calculate the values
    total_unique_ids = df["internal supplier id or vendor number"].nunique()

    # Define the columns that need to be checked for gaps
    address_columns = [
        "address (street name and number)",
        "address (city)",
        "address (state/province)",
        "address (country)",
        "address (zip code / postal code)"
    ]

    # Define the condition to identify gaps (blanks or "n/a" values)
    gap_conditions = lambda x: bool(re.match(r'^\s*$', str(x))) or str(x).strip().lower() == 'n/a'

    # Group by "internal supplier id or vendor number" and apply gap checks to each group
    unique_id_groups = df.groupby("internal supplier id or vendor number")

    # Initialize counters for gaps in each address column
    street_address_gaps = 0
    city_gaps = 0
    state_gaps = 0
    country_gaps = 0
    zip_code_gaps = 0

    # Check gaps within each group for each address component
    for _, group in unique_id_groups:
        # Check for gaps in each address component and count them
        if group["address (street name and number)"].apply(gap_conditions).any():
            street_address_gaps += 1
        if group["address (city)"].apply(gap_conditions).any():
            city_gaps += 1
        if group["address (state/province)"].apply(gap_conditions).any():
            state_gaps += 1
        if group["address (country)"].apply(gap_conditions).any():
            country_gaps += 1
        if group["address (zip code / postal code)"].apply(gap_conditions).any():
            zip_code_gaps += 1

    # Count unique addresses with gaps (any column has a gap)
    rows_with_any_gap = unique_id_groups.apply(
        lambda x: any(x[col].apply(gap_conditions).any() for col in address_columns)
    ).sum()

    # Append calculated data
    data_row = [total_unique_ids, rows_with_any_gap, street_address_gaps, city_gaps, state_gaps, country_gaps, zip_code_gaps]
    ws.append(data_row)

    # Insert and calculate B3 value as a percentage
    if total_unique_ids:
        ws["B3"] = rows_with_any_gap / total_unique_ids
    else:
        ws["B3"] = 0

    # Save as a separate file
    output_gaps_file = "Addr_Gaps_Start.xlsx"
    wb.save(output_gaps_file)
    return output_gaps_file

# COPY THE TEMPORARY PRE PROCESSING DATA IN ADDRESS GAPS TAB
def copy_addr_gaps_data_to_main(main_workbook_file, output_gaps_file):
    # Load the "Addr. Gaps to Start" file and read data from rows 2 and 3
    gaps_wb = load_workbook(output_gaps_file)
    gaps_ws = gaps_wb["Addr. Gaps to Start"]
    rows_to_copy = list(gaps_ws.iter_rows(min_row=2, max_row=3, values_only=True))

    # Load the main workbook and add data to "Address Gaps" tab at rows 5 and 6
    main_wb = load_workbook(main_workbook_file)
    if "Address Gaps" not in main_wb.sheetnames:
        main_ws = main_wb.create_sheet("Address Gaps")
    else:
        main_ws = main_wb["Address Gaps"]

    # Add headers to "Address Gaps" if needed
    headers = ["Total Unique Supplier IDs", "Number of Unique Addresses with Gaps", "Street Address Gaps", "City Gaps", "State Gaps", "Country Gaps", "Zip Code Gaps"]
    if main_ws.max_row < 5:
        main_ws.append(headers)

    # Insert the copied data into rows 5 and 6
    for i, row_data in enumerate(rows_to_copy, start=5):
        for j, value in enumerate(row_data, start=1):
            main_ws.cell(row=i, column=j, value=value)

    # Save the updated main workbook
    main_wb.save(main_workbook_file)

# TAKE THE ADDRESS GAP DATAPOINTS AND CREATE A CLEANING SUMMARY TAB
def add_address_gaps_and_cleaning_summary(main_output_file, df):
    # Load the main workbook and add the "Cleaning Summary" sheet
    wb = load_workbook(main_output_file)
    cleaning_summary_sheet = wb.create_sheet(title="Cleaning Summary")

    # Define headers and write them
    headers = ["Category", "Before Processing", "Post Processing", "Improvement", "Invalid Values"]
    cleaning_summary_sheet.append(headers)

    # Categories for summary
    categories = [
        "Total Unique Supplier IDs", "Count of Unique Addresses with Gaps",
        "Percentage of Unique Addresses with Gaps", "Street Address Gaps", "City Gaps",
        "State Gaps", "Country Gaps", "Zip Code Gaps", "Count of Special Characters in Names",
        "Total Unique Names Cleaned", "Street Address (less than 5 alphanumeric)", "City (less than 3 alphanumeric)",
        "State (less than 2 alphanumeric)", "Country (less than 2 alphanumeric)", "Zip Code (less than 4 alphanumeric)"
    ]

    # Fetch values from Address Gaps tab
    address_gaps_sheet = wb["Address Gaps"]
    before_processing_values = [
        address_gaps_sheet["A5"].value, address_gaps_sheet["B5"].value, address_gaps_sheet["B6"].value,
        address_gaps_sheet["C5"].value, address_gaps_sheet["D5"].value, address_gaps_sheet["E5"].value,
        address_gaps_sheet["F5"].value, address_gaps_sheet["G5"].value
    ]

    # Additional calculations for special characters
    unedited_name_special_count = df["unedited company name"].str.count(r"[!@$%^*()\[\]?;/\\]").sum()
    unedited_name_special_rows = df["unedited company name"].str.contains(r"[!@$%^*()\[\]?;/\\]").sum()
    before_processing_values.extend([unedited_name_special_count, unedited_name_special_rows])

    # Processed values from the cleaned "company name" column
    cleaned_name_special_count = df["company name"].str.count(r"[!@$%^*()\[\]?;/\\]").sum()
    cleaned_name_special_rows = df["company name"].str.contains(r"[!@$%^*()\[\]?;/\\]").sum()
    post_processing_values = [
        address_gaps_sheet["A2"].value, address_gaps_sheet["B2"].value, address_gaps_sheet["B3"].value,
        address_gaps_sheet["C2"].value, address_gaps_sheet["D2"].value, address_gaps_sheet["E2"].value,
        address_gaps_sheet["F2"].value, address_gaps_sheet["G2"].value,
        cleaned_name_special_count, cleaned_name_special_rows
    ]

    # Append placeholder values for new "Invalid Values" categories
    before_processing_values.extend(["-"] * 5)
    post_processing_values.extend(["-"] * 5)

    # Calculate invalid values for new categories
    invalid_values = [
        "-", "-", "-", "-", "-", "-", "-", "-", "-", "-",
        df["address (street name and number)"].str.replace(r"\W", "", regex=True).str.len().lt(5).sum(),
        df["address (city)"].str.replace(r"\W", "", regex=True).str.len().lt(3).sum(),
        df["address (state/province)"].str.replace(r"\W", "", regex=True).str.len().lt(2).sum(),
        df["address (country)"].str.replace(r"\W", "", regex=True).str.len().lt(2).sum(),
        df["address (zip code / postal code)"].str.replace(r"\W", "", regex=True).str.len().lt(4).sum()
    ]

    # Apply categories, values, and improvement calculations to the Cleaning Summary sheet
    for i, category in enumerate(categories):
        cleaning_summary_sheet.cell(row=i + 2, column=1, value=category)  # Category
        before_cell = cleaning_summary_sheet.cell(row=i + 2, column=2, value=before_processing_values[i])  # Before
        post_cell = cleaning_summary_sheet.cell(row=i + 2, column=3, value=post_processing_values[i])  # Post
        improvement_cell = cleaning_summary_sheet.cell(row=i + 2, column=4)  # Improvement
        invalid_cell = cleaning_summary_sheet.cell(row=i + 2, column=5, value=invalid_values[i])  # Invalid Values

        # Calculate improvement
        if category in ["Count of Special Characters in Names", "Total Unique Names Cleaned"]:
            improvement_cell.value = before_processing_values[i] - post_processing_values[i]
        elif isinstance(before_processing_values[i], (int, float)) and isinstance(post_processing_values[i], (int, float)):
            improvement_cell.value = before_processing_values[i] - post_processing_values[i]
        else:
            improvement_cell.value = "-"

        # Right-align "-" placeholders
        for cell in [before_cell, post_cell, improvement_cell, invalid_cell]:
            if cell.value == "-":
                cell.alignment = Alignment(horizontal="right")

    # Apply conditional formatting to make positive values green in the Improvement column
    green_font_rule = CellIsRule(operator='greaterThan', formula=['0'], font=Font(color="008000"))
    cleaning_summary_sheet.conditional_formatting.add(f'D2:D{cleaning_summary_sheet.max_row}', green_font_rule)

    # Additional styling for the header row and Invalid Values inputs (as before)
    for cell in cleaning_summary_sheet["1:1"]:
        cell.font = Font(bold=True)
        cell.fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
    for row in cleaning_summary_sheet.iter_rows(min_row=2, max_row=len(categories) + 1, min_col=1, max_col=3):
        for cell in row:
            cell.fill = PatternFill(start_color="CCFFFF", end_color="CCFFFF", fill_type="solid")
    improvement_header_cell = cleaning_summary_sheet["D1"]
    improvement_header_cell.font = Font(bold=True)
    improvement_header_cell.fill = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")
    for row in cleaning_summary_sheet.iter_rows(min_row=2, max_row=cleaning_summary_sheet.max_row, min_col=4, max_col=4):
        for cell in row:
            cell.fill = PatternFill(start_color="FFD580", end_color="FFD580", fill_type="solid")
    cleaning_summary_sheet["E1"].font = Font(bold=True)
    cleaning_summary_sheet["E1"].fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    light_yellow_fill = PatternFill(start_color="FFFFE0", end_color="FFFFE0", fill_type="solid")
    for row in cleaning_summary_sheet.iter_rows(min_row=2, max_row=cleaning_summary_sheet.max_row, min_col=5, max_col=5):
        for cell in row:
            cell.fill = light_yellow_fill

    # Column widths (as before)
    cleaning_summary_sheet.column_dimensions['A'].width = 33
    cleaning_summary_sheet.column_dimensions['B'].width = 16
    cleaning_summary_sheet.column_dimensions['C'].width = 16
    cleaning_summary_sheet.column_dimensions['D'].width = 15
    cleaning_summary_sheet.column_dimensions['E'].width = 15

    # Set percentage format for "Percentage of Unique Addresses with Gaps" cells
    cleaning_summary_sheet["B4"].number_format = "0.00%"  # Before Processing
    cleaning_summary_sheet["C4"].number_format = "0.00%"  # Post Processing
    cleaning_summary_sheet["D4"].number_format = "0.00%"  # Improvement

    # Move Address Gaps sheet to the end and color it dark grey
    address_gaps_sheet = wb["Address Gaps"]
    wb._sheets.remove(address_gaps_sheet)  # Remove from current position
    wb._sheets.append(address_gaps_sheet)  # Re-append it to move it to the end
    address_gaps_sheet.sheet_properties.tabColor = "404040"  # Dark grey color

    # Save changes
    wb.save(main_output_file)

# Generate a list of common names using the 'names' library
def get_common_names_from_names_lib(count=5000):
    common_names = set()
    while len(common_names) < count:
        common_names.add(names.get_first_name().lower())
    return common_names

def safe_str(value):
    """Converts the given value to a string if it's not already one."""
    if not isinstance(value, str):
        try:
            return str(value)
        except Exception:
            return ''
    return value

# Fetch the list of Nasdaq 100 companies using yfinance
def get_nasdaq_100_companies():
    nasdaq_100_tickers = [
        'AAPL', 'MSFT', 'AMZN', 'TSLA', 'GOOGL', 'GOOG', 'META', 'NVDA', 'PEP', 'NFLX',
        'ADBE', 'CSCO', 'INTC', 'CMCSA', 'PYPL', 'QCOM', 'TXN', 'HON', 'AMGN', 'SBUX',
        'COST', 'MRK', 'GILD', 'BKNG', 'VRTX', 'ATVI', 'AVGO', 'AMD', 'REGN', 'BIIB',
        'LRCX', 'FISV', 'MU', 'SIRI', 'KLAC', 'CRWD', 'CDNS', 'PANW', 'IDXX', 'SNPS',
        'MRNA', 'WDAY', 'ASML', 'TEAM', 'ALGN', 'KDP', 'ADI', 'DXCM', 'OKTA', 'NXPI',
        'ZS', 'MRVL', 'XLNX', 'EXC', 'ANSS', 'MCHP', 'EA', 'FANG', 'AMAT', 'CPRT',
        'ENPH', 'SWKS', 'CTAS', 'FTNT', 'BMRN', 'MTCH', 'LULU', 'FAST', 'ILMN', 'ROST',
        'PAYX', 'EBAY', 'CHKP', 'BIDU', 'MAR', 'VRSK', 'SGEN', 'VRSN', 'PTON', 'PINS',
        'AZN', 'ETSY', 'SIRI', 'XEL', 'ABNB', 'DDOG', 'DDOG', 'CRWD', 'ZM', 'DOCU',
        'RIVN', 'DDOG', 'DDOG', 'TEAM', 'ABNB', 'BMRN', 'PDD', 'JD', 'BIDU', 'LCID',
        'PDD', 'BIDU', 'DDOG', 'CHWY', 'ZG', 'SNAP', 'DOCU', 'UBER', 'FDX', 'MRNA',
        'PDD', 'LCID'
    ]
    tickers_data = yf.Tickers(nasdaq_100_tickers)
    company_names = []
    for ticker in nasdaq_100_tickers:
        try:
            info = tickers_data.tickers[ticker].info
            if 'shortName' in info:
                company_names.append(info['shortName'])
        except HTTPError as e:
            print(f"HTTPError for ticker {ticker}: {e}")
        except Exception as e:
            print(f"Error for ticker {ticker}: {e}")
    return company_names

# Define a list of company names that are also person names
def get_company_names_person_names():
    return ["Ralph Lauren", "Sherwin Williams","Thermo Fisher","JP Morgan","Mary Kay", "Duane Reade", "Tommy Hilfiger", "Wendy's", "Martha Stewart", "Ford", "Hugo Boss", "Victoria's Secret", "Fisher-Price"]

# Remove spacing from country name
def clean_country_name(country_name):
    # Ensure the country_name is treated as a string
    country_name = str(country_name)

    # Check if the value is numeric or contains only digits
    if country_name.isdigit():
        return ''  # Exclude number-only values

    # Remove punctuation from the string
    country_name = country_name.translate(str.maketrans('', '', string.punctuation))

    return country_name

# Initialize the geocoder for a specific country (e.g., US)
nomi = pgeocode.Nominatim('us')

# Add a leading zero to 4-digit ZIP codes if the country is United States
def format_us_zip_codes(df):
    if 'address (zip code / postal code)' in df.columns and 'address (country)' in df.columns:
        # Ensure the ZIP code column is treated as a string type
        df['address (zip code / postal code)'] = df['address (zip code / postal code)'].astype(str)

        # Set of acceptable country values
        us_variants = {"united states", "us", "u.s.", "usa", "u.s.a.", "united states of america"}

        def add_leading_zero(row):
            zip_code = row['address (zip code / postal code)']
            country = str(row['address (country)']).strip().lower()

            # Check if the ZIP code has a length of 4 and the country matches any US variant
            if zip_code.isdigit() and len(zip_code) == 4 and country in us_variants:
                print(f"Adding leading zero to ZIP code: {zip_code} for country: {country}")
                return f"0{zip_code}"
            return zip_code

        # Apply the function to the DataFrame
        df['address (zip code / postal code)'] = df.apply(add_leading_zero, axis=1)

    return df

# Function to get city and state from ZIP code
def get_city_state_from_zip(zip_code):
    # Ensure the ZIP code is a string and read only the first 5 characters
    zip_code = str(zip_code)[:5]

    # Check if the truncated ZIP code is valid (should be all digits and 5 characters long)
    if zip_code.isdigit() and len(zip_code) == 5:
        location = nomi.query_postal_code(zip_code)
        if pd.notna(location.place_name) and pd.notna(location.state_name):
            city = location.place_name.split(',')[0]  # To handle multiple city names in the data
            state = location.state_name
            return city, state
    return None, None

# Fill missing city/state based on ZIP code using pgeocode
def fill_missing_city_state(df):
    if 'address (zip code / postal code)' in df.columns:
        for idx, row in df.iterrows():
            if not row['address (city)'] or not row['address (state/province)']:  # Check if city or state is blank
                city, state = get_city_state_from_zip(row['address (zip code / postal code)'])
                if city and not row['address (city)']:
                    df.at[idx, 'address (city)'] = city
                if state and not row['address (state/province)']:
                    df.at[idx, 'address (state/province)'] = state
    return df

def normalize_country_name(country_name):
    country_name = country_name.strip().lower()

    # Handle specific cases to return pycountry values
    specific_cases = {
        'united states of america': 'United States',
        'usa': 'United States',
        'u.s.a.': 'United States',
        'us': 'United States',
        'united states': 'United States',
        'korea, republic of': 'Korea, Republic of',
        'south korea': 'Korea, Republic of',
        'czechia': 'Czechia',
        'czech republic': 'Czechia',
        'vietnam': 'Viet Nam',
        'taiwan': 'Taiwan, Province of China',
        'taiwan, province of china': 'Taiwan, Province of China',
        'turkey': 'Türkiye',
        'russia': 'Russian Federation',
        'russian federation': 'Russian Federation'
    }

    if country_name in specific_cases:
        return specific_cases[country_name]

    # Default pycountry logic
    for country in pycountry.countries:
        if country_name == country.name.lower() or country_name == country.alpha_2.lower() or country_name == country.alpha_3.lower():
            return country.name

    return "ERROR"  # Return "ERROR" if no match is found

def read_excel_file(file_path, sheet_name="Vendor Master"):
    # Load the Excel file, checking if `sheet_name` is specified or not
    if sheet_name:
        df = pd.read_excel(file_path, sheet_name=sheet_name)
    else:
        # Load all sheets as a dictionary and select the first sheet if `sheet_name=None`
        sheets = pd.read_excel(file_path, sheet_name=None)
        first_sheet_name = list(sheets.keys())[0]  # Get the first sheet name
        df = sheets[first_sheet_name]  # Use the first sheet as the DataFrame

    df.columns = df.columns.str.lower()  # Convert column names to lowercase
    df = df.fillna('')  # Fill NaN values with empty strings

    # Ensure the ZIP code column is treated as a string and remove any ".0" suffix
    if "address (zip code / postal code)" in df.columns:
        df["address (zip code / postal code)"] = df["address (zip code / postal code)"].astype(str)
        df["address (zip code / postal code)"] = df["address (zip code / postal code)"].str.replace(r'\.0$', '', regex=True)

    # Convert specific columns to numeric if needed for calculations (example: spend amount)
    numeric_columns = ['spend amount*', 'other_numeric_column']  # replace with actual numeric column names
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')  # Convert with NaN for non-numeric entries

    print(f"Columns in input file: {df.columns.tolist()}")  # Debug: Print columns in input file
    return df

# Normalize state/province names using the us library
def normalize_state_province(df):
    if "address (state/province)" in df.columns:
        def normalize_state(state_name):
            if isinstance(state_name, str):
                state_name = state_name.strip()
                state = us.states.lookup(state_name)
                if state:
                    return state.abbr  # Normalize to two-letter abbreviation
            return state_name  # Return as is if no match found

        df["address (state/province)"] = df["address (state/province)"].apply(normalize_state)
    return df

def add_columns(df):
    if 'internal supplier id' not in df.columns and 'internal supplier id or vendor number' in df.columns:
        df['internal supplier id'] = df['internal supplier id or vendor number']
    elif 'internal supplier id' not in df.columns:
        df['internal supplier id'] = ''
    if 'spendcountry' not in df.columns:
        df['spendcountry'] = ''
    if 'code' not in df.columns:
        df['code'] = ''
    if 'sanctioned country' not in df.columns:
        df['sanctioned country'] = ''
    return df

def check_required_columns(df):
    required_columns = {
        'street address': '',
        'city': '',
        'state': '',
        'country': '',
        'zip code': ''
    }

    for col in df.columns:
        if isinstance(col, str):  # Ensure the column name is a string
            if 'street name and number' in col.lower():
                required_columns['street address'] = col
            elif 'city' in col.lower():
                required_columns['city'] = col
            elif 'state' in col.lower():
                required_columns['state'] = col
            elif 'address (country)' in col.lower():
                required_columns['country'] = col
            elif 'zip' in col.lower() or 'postal' in col.lower():
                required_columns['zip code'] = col

    print(f"Column mappings: {required_columns}")  # Debug: Print column mappings
    return required_columns

def create_or_clean_complete_address(df, col_mappings):
    """
    Creates or cleans the 'Complete Full Address' based on the presence of address components.
    If at least 2 address components have values, it will create a new 'Complete Full Address'.
    If fewer than 2 address components have values, it will retain or clean the existing 'Complete Full Address'.
    """

    required_columns = [
        col_mappings.get("street address", ""),
        col_mappings.get("city", ""),
        col_mappings.get("state", ""),
        col_mappings.get("zip code", ""),
        col_mappings.get("country", "")
    ]

    required_columns = [col for col in required_columns if col]

    # Check if required columns exist in the dataframe
    if not required_columns:
        print("Required address columns are missing. Exiting script.")
        return df

    def apply_address_logic(row):
        non_empty_components = [
            row[col] for col in required_columns if pd.notna(row[col]) and row[col] != ''
        ]

        # If fewer than 2 address components have values, keep or clean the existing 'Complete Full Address'
        if len(non_empty_components) < 2:
            if 'complete full address' in row and row['complete full address']:
                return clean_text(row['complete full address'])  # Clean existing 'Complete Full Address'
            else:
                return ''  # Return empty string if 'Complete Full Address' is also empty

        # If 2 or more address components have values, create a new 'Complete Full Address'
        else:
            return join_address_components(row, required_columns)


    # Apply the logic to each row
    df['complete full address'] = df.apply(apply_address_logic, axis=1)

    print("Complete full address updated.")  # Debug: Indicate completion
    print(df["complete full address"].head())  # Debug: Print the first few complete addresses
    return df

def join_address_components(row, required_columns):
    # Convert each part to a string, handling cases where floats are actually integers to prevent ".0"
    parts = [
        str(int(row[col])) if isinstance(row[col], float) and row[col].is_integer() else str(row[col]).strip()
        for col in required_columns if pd.notna(row[col]) and str(row[col]).strip() != ''
    ]
    return ', '.join(parts)

def create_complete_address(df, col_mappings):
    required_columns = [col_mappings["street address"], col_mappings["city"], col_mappings["state"], col_mappings["zip code"], col_mappings["country"]]
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    df["complete full address"] = df.apply(lambda row: join_address_components(row, required_columns, col_mappings), axis=1)
    return df

def clean_text(text):
    text = safe_str(text)  # Apply the safe_str function first
    if isinstance(text, str):
        text = re.sub(r"\(.*?\)|\[.*?\]", "", text)  # Remove brackets
        text = re.sub(r"[!@$%^*()\[\]?;/\\]", "", text)  # Remove special characters
        text = re.sub(r"^[^A-Za-z0-9]+", "", text)  # Trim leading non-alphanumeric characters
        text = re.sub(r"\s*,\s*", ", ", text)  # Ensure single space after commas
        text = re.sub(r", ,", ", ", text)  # Replace ", ," with ", "
        text = re.sub(r",,", ", ", text)  # Replace ",," with ", "
        text = re.sub(r",  ,", ", ", text)  # Replace ",  ," with ", "
        text = re.sub(r"\s+", " ", text)  # Replace multiple spaces with a single space
    return text


def clean_columns(df):
    if "company name" in df.columns:
        df["unedited company name"] = df["company name"]
        df["company name"] = df["company name"].apply(clean_text)

    df["complete full address"] = df["complete full address"].apply(clean_text)
    return df

# Normalize state/province names using the us library to full state names
def normalize_state_province(df):
    if "address (state/province)" in df.columns:
        def normalize_state(state_name):
            if isinstance(state_name, str):
                state_name = state_name.strip()
                state = us.states.lookup(state_name)
                if state:
                    return state.name  # Normalize to full state name
            return state_name  # Return as is if no match found

        df["address (state/province)"] = df["address (state/province)"].apply(normalize_state)
    return df


# Update country based on city and state/province for USA
def update_country_based_on_city_state(df):
    """
    Updates the 'address (country)' column based on the city and state/province columns.
    """
    for index, row in df.iterrows():
        # Safely convert city and state values to strings
        city = safe_str(row.get('address (city)', '')).strip().lower()
        state_abbr = safe_str(row.get('address (state/province)', '')).strip().upper()

        # Check if the country is missing but city and state are present
        if not row.get('address (country)', '').strip() and city and state_abbr:
            # Check if the state abbreviation is valid
            if us.states.lookup(state_abbr):
                df.at[index, 'address (country)'] = "USA"
    return df


def update_code_based_on_common_names(df, common_names1):
    exclude_keywords = [
        "department", "llc", "pllc", "inc.", "incorporated", "ltd", "company",
        "corp.", "corporation", "foundation", "charity", "non-profit", "campaign",
        "county", "city", "town", "state", "govt", "government", "organization",
        "equipment", "north", "west", "east", "south", "service", "society",
        "comms.", "communications", "chevrolet", "audi", "nissan", "jeep",
        "dodge", "gmc", "cadillac", "ford", "honda", "hyundai", "group", "royal",
        "tesla", "dealership", "automotive", "toyota", "motors", "creative", " corp",
        "mitsubishi", "international", " inc", "library","school", "llp","associate",
        "limited","lp"," pc","securities","1","2","3","4","5","ventures","traders",
        "america","photography","S.A.S","hotel", "UNIVERSITE","dmv","energy","finance",
        "paris","france","storage","marriott","hilton","management", "mgmt",
        "general", "association", "AMEX","Council","Canada","Transportation","electronic",
        "system","material","citizen","consultant","technology", "broker", "florida","program",
        "production","at&t","contract","comptroller","maryland","university","talent","electric",
        "fitness","frontier", "&", "family","advocacy","product","mutual","illinois","public","wireless",
        "business","towing","recovery","all star","japan","workshop","enterprise","executive","global",
        "adjuster","village","natural","design","investigation","neighbor","ambulance",
        "software","recycling"," US ","batteries","sheriff","elect","commonwealth", "computers",
        "consulting","power","chapter","cloud","beach","for","first","hyatt","restaurant","repossession",
        "solutions","adhesives","courtyard","authority","chicago","wrecker","of","committee","bank",
        "chapter","PAC","station","office","moving","college","nova scotia","specialties",
        "chase ink","co.","supply","services","catering","assoc.","technologies","optical",
        "metal","invest","hospital","laboratories","clinic","drug","automation","instruments",
        "EMC","GPS","health","IMA","innovation","store","universidad","genomics","biomedical",
        "research","conferences","labs","scientific","San Diego","learning","laboratory",
        "controls","puerto rico","institute","srl","universita","and","medicine","medicines",
        "conf","motion","national","china","argentina","industria","india","bioanalytics","coalition",
        "x ray","mechanical","microplates","interpretable","columbus","gmbh","taxi","+",
        "Marie Claire", "repairs",",","beijing","sciences","biologics","centre","farmacia",
        "disability", "chemicals", "science", "Sherwin Williams", "Dell Computer", "Dell AB",
        "Dell SP","solicitors", "Communication","magic","cardio","Capital", "BV", "plastics",
        "chemical","papers", "fragrances","ingredients","cosmetics","s.r.o.","s.r.l.","sales", " - ",
        "gobierno", "pizzeria", "balloon", "rental", "dental", "pharmacy", "eatery", "flowers", "mediterranean",
        "installation", "gallery", "genetics", "sro", "trucking", "contemporary", "jewelers", "pizza", "fashioned",
        "yoga", "imports", "valet", "sports", "training", "center", "retirement", "pharmaceuticals", "the",
        "intercontinental", "events", "cleaners", "financial", "neurogenetics","catholic", "printing", "challenge",
        "recruiting", "bio-medical", "church", "market", "reporting", "radiology", "entertainment", "dermatology",
        "by", "fence", "community", "cable", "development", "construction", "security", "lodge", "craft", "studio",
        "administration", "medical", "fabricating", "donut", "living", "area", "district", "logistics", "projects",
        "6","7","8","9","Academic", "Academy", "Agency", "AI", "Akron", "House", "Announcer", "Apartments", "Architect",
        "Architects", "Art", "Athletics", "Avenue", "Awards", "Ballroom", "Behavioral", "Binders", "Biologicals",
        "Bionetworks","Bioresources", "Biotec", "Biotech", "Books", "Bookseller", "Booksellers", "Boston", "Box",
        "Broadway", "Bus", "Cafe", "Camp", "Campus", "Cantina", "Carpets", "Cellular", "Central", "Centrum", "Childress",
        "Clarion", "Club", "Co", "Coaching","Commons", "Companies", "Composites", "Consolidated", "Cost", "Courier",
        "Curves", "Custom", "Brass", "DBA", "Deed", "Delta", "Omega", "Diesel", "Filtration", "Directions",
        "Distribution", "Diversity", "Action", "Drive", "Paint", "Echonous", "Editing", "Editorial", "Education",
        "Latino", "Elevator", "Expo", "Express", "Expressions", "Collective", "Films", "Flooring", "Food", "Furniture",
        "Engineering", "Galeria", "Garage", "Garden", "Gift", "Glass", "Golden", "Golf", "Gulfstream", "Hardware",
        "Hatchery", "Hidden", "Springs", "Holdings", "House", "impress", "industries", "Instrument", "Insurance",
        "Integration", "interiors", "INTL", "Kahoot", "Kitchen", "Lattitude", "Laundry", "laws", "leasing",
        "in", "percussion", "sustainability", "ltp", "agency", "maps", "marine", "media", "medicina", "memorial",
        "memory", "military", "ml", "movers", "museum", "music", "my", "nai", "naked", "nautica", "nepal",
        "network", "neurocare", "p.a.", "newark", "nutrition", "ohio", "travel", "operation", "orange", "organize",
        "pa", "painter", "partners","photo","photographics", "photos", "pinch", "plc", "petals", "presents",
        "press", "produce", "publishing", "pulse", "project", "quick", "quintet", "records", "refractories", "regional",
        "renovations", "rlty", "room", "rooted", "roots", "scene", "scissors", "australia", "skyline", "mind", "sound",
        "speaking", "speaks", "stinger", "stock", "trade", "textile", "trans", "studies", "trust", "united", "USA",
        "videos", "visuals", "wholesale", "signs", "worship", "writing", "zoo"
    ]
    company_names_person_names = get_company_names_person_names()

    def update_code(row):
        # Ensure company_name is always treated as a string
        company_name = safe_str(row['company name']).lower()  # Use safe_str to handle non-string values

        # Check for exclusion based on exact company names first
        if company_name in company_names_person_names:
            print(f"Excluding company name: {company_name} from 'P' code assignment due to exact match exclusion.")
            return row['code']

        # Only proceed if company_name is not empty and contains one or two spaces
        if row['code'] == '' and company_name.count(' ') in [1, 2]:  # Ensure proper indentation here
            # Ensure each part is treated as a string
            parts = [safe_str(part).lower() for part in company_name.split()]

            # Check if any part matches common names and exclusion keywords
            if any(part in common_names1 for part in parts):
                # Make sure we are only checking strings against keywords
                if not any(keyword.lower() in company_name for keyword in exclude_keywords):
                    print(f"Assigning 'P' to company name: {company_name}")
                    return 'P'

        return row['code']

    if "company name" in df.columns:
        df["code"] = df.progress_apply(update_code, axis=1)

    return df



def update_code_based_on_ner(df, common_names1, company_names_person_names):
    ner_pipeline = pipeline("token-classification", model="dbmdz/bert-large-cased-finetuned-conll03-english")

    exclude_keywords = [
        "department", "llc", "pllc", "inc.", "incorporated", "ltd", "company",
        "corp.", "corporation", "foundation", "charity", "non-profit", "campaign",
        "county", "city", "town", "state", "govt", "government", "organization",
        "equipment", "north", "west", "east", "south", "service", "society",
        "comms.", "communications", "chevrolet", "audi", "nissan", "jeep",
        "dodge", "gmc", "cadillac", "ford", "honda", "hyundai", "group", "royal",
        "tesla", "dealership", "automotive", "toyota", "motors", "creative", " corp",
        "mitsubishi", "international", " inc", "library", "school", "llp", "associate",
        "limited", "lp", "pc", "securities", "1", "2", "3", "4", "5", "ventures", "traders",
        "america", "photography", "S.A.S", "hotel", "UNIVERSITE", "dmv", "energy", "finance",
        "paris", "france", "storage", "marriott", "hilton", "management", "mgmt",
        "general", "association", "AMEX", "Council", "Canada", "Transportation", "electronic",
        "system", "material", "citizen", "consultant", "technology", "broker", "florida", "program",
        "production", "at&t", "contract", "comptroller", "maryland", "university", "talent", "electric",
        "fitness", "frontier", "&", "family", "advocacy", "product", "mutual", "illinois", "public", "wireless",
        "business", "towing", "recovery", "all star", "japan", "workshop", "enterprise", "executive", "global",
        "adjuster", "village", "natural", "design", "investigation", "neighbor", "ambulance",
        "software", "recycling", "US", "batteries", "sheriff", "elect", "commonwealth", "computers",
        "consulting", "power", "chapter", "cloud", "beach", "for", "first", "hyatt", "restaurant", "repossession",
        "solutions", "adhesives", "courtyard", "authority", "chicago", "wrecker", "of", "committee", "bank",
        "chapter", "PAC", "station", "office", "moving", "college", "nova scotia", "specialties",
        "chase ink", "co.", "supply", "services", "catering", "assoc.", "technologies", "optical",
        "metal", "invest", "hospital", "laboratories", "clinic", "drug", "automation", "instruments",
        "EMC", "GPS", "health", "IMA", "innovation", "store", "universidad", "genomics", "biomedical",
        "research", "conferences", "labs", "scientific", "San Diego", "learning", "laboratory",
        "controls", "puerto rico", "institute", "srl", "universita", "and", "medicine", "medicines",
        "conf", "motion", "national", "china", "argentina", "industria", "india", "bioanalytics", "coalition",
        "x ray", "mechanical", "microplates", "interpretable", "columbus", "gmbh", "taxi", "+",
        "Marie Claire", "repairs", ",", "beijing", "sciences", "biologics", "centre", "farmacia",
        "disability", "chemicals", "science", "Sherwin Williams", "Dell Computer", "Dell AB",
        "Dell SP", "solicitors", "Communication", "magic", "cardio", "Capital", "BV","plastics",
        "chemical","papers", "fragrances","ingredients","cosmetics","s.r.o.","s.r.l.","sales", " - ",
        "gobierno", "pizzeria", "balloon", "rental", "dental", "pharmacy", "eatery", "flowers", "mediterranean",
        "installation", "gallery", "genetics", "sro", "trucking", "contemporary", "jewelers", "pizza", "fashioned",
        "yoga", "imports", "valet", "sports", "training", "center", "retirement", "pharmaceuticals", "the",
        "intercontinental", "events", "cleaners", "financial", "neurogenetics","catholic", "printing", "challenge",
        "recruiting", "bio-medical", "church", "market", "reporting", "radiology", "entertainment", "dermatology",
        "by", "fence", "community", "cable", "development", "construction", "security", "lodge", "craft", "studio",
        "administration", "medical", "fabricating", "donut", "living", "area", "district", "logistics", "projects",
        "6","7","8","9","Academic", "Academy", "Agency", "AI", "Akron", "House", "Announcer", "Apartments", "Architect",
        "Architects", "Art", "Athletics", "Avenue", "Awards", "Ballroom", "Behavioral", "Binders", "Biologicals",
        "Bionetworks","Bioresources", "Biotec", "Biotech", "Books", "Bookseller", "Booksellers", "Boston", "Box",
        "Broadway", "Bus", "Cafe", "Camp", "Campus", "Cantina", "Carpets", "Cellular", "Central", "Centrum", "Childress",
        "Clarion", "Club", "Co", "Coaching","Commons", "Companies", "Composites", "Consolidated", "Cost", "Courier",
        "Curves", "Custom", "Brass", "DBA", "Deed", "Delta", "Omega", "Diesel", "Filtration", "Directions",
        "Distribution", "Diversity", "Action", "Drive", "Paint", "Echonous", "Editing", "Editorial", "Education",
        "Latino", "Elevator", "Expo", "Express", "Expressions", "Collective", "Films", "Flooring", "Food", "Furniture",
        "Engineering", "Galeria", "Garage", "Garden", "Gift", "Glass", "Golden", "Golf", "Gulfstream", "Hardware",
        "Hatchery", "Hidden", "Springs", "Holdings", "House", "impress", "industries", "Instrument", "Insurance",
        "Integration", "interiors", "INTL", "Kahoot", "Kitchen", "Lattitude", "Laundry", "laws", "leasing",
        "in", "percussion", "sustainability", "ltp", "agency", "maps", "marine", "media", "medicina", "memorial",
        "memory", "military", "ml", "movers", "museum", "music", "my", "nai", "naked", "nautica", "nepal",
        "network", "neurocare", "p.a.", "newark", "nutrition", "ohio", "travel", "operation", "orange", "organize",
        "pa", "painter", "partners","photo","photographics", "photos", "pinch", "plc", "petals", "presents",
        "press", "produce", "publishing", "pulse", "project", "quick", "quintet", "records", "refractories", "regional",
        "renovations", "rlty", "room", "rooted", "roots", "scene", "scissors", "australia", "skyline", "mind", "sound",
        "speaking", "speaks", "stinger", "stock", "trade", "textile", "trans", "studies", "trust", "united", "USA",
        "videos", "visuals", "wholesale", "signs", "worship", "writing", "zoo"
    ]

    def update_code(row):
        company_name = safe_str(row['company name']).lower()  # Use safe_str to ensure company_name is a string

        # Ensure the company name contains one or two spaces
        if company_name.count(' ') not in [1, 2]:
            return row['code']  # Skip further processing if the condition isn't met

        # Check for named entities using Hugging Face model
        ner_results = ner_pipeline(company_name)

        # Check if any entity is identified as a person
        if any(entity['entity'].startswith('I-PER') for entity in ner_results):
            # Check for exclusion based on exact company names first
            if company_name in company_names_person_names:
                print(f"Excluding company name: {company_name} from 'P' code assignment due to exact match exclusion.")
                return row['code']
            # Check for exclusion keywords
            if not any(keyword.lower() in company_name for keyword in exclude_keywords):
                print(f"Assigning 'P' to company name: {company_name}")
                return 'P'

        # Apply common names logic
        parts = company_name.split()
        if any(part.lower() in common_names1 for part in parts):
            # Check for exclusion keywords
            if not any(keyword.lower() in company_name for keyword in exclude_keywords):
                print(f"Assigning 'P' to company name: {company_name}")
                return 'P'

        # Check for MD/M.D. logic
        if ", m.d." in company_name or ", md" in company_name:
            return 'P'

        return row['code']

    if "company name" in df.columns:
        df["code"] = df.progress_apply(update_code, axis=1)
    return df



def remove_p_based_on_combined_names(df, combined_names):
    def remove_p(row):
        company_name = safe_str(row['company name'])  # Ensure company_name is a string
        if row['code'] == 'P' and company_name in combined_names:
            return ''
        return row['code']

    if "company name" in df.columns:
        df["code"] = df.apply(remove_p, axis=1)
    return df

def update_code_based_on_contact_email(df, client_domain):
    if "contact email" in df.columns:
        def check_domain(row):
            if row['code'] == 'P':
                contact_email = safe_str(row.get("contact email", "")).lower()  # Ensure contact_email is a string
                if client_domain in contact_email:
                    return 'A'
            return row['code']

        df["code"] = df.apply(check_domain, axis=1)
    return df

def update_code_based_on_company_name(df):
    if "company name" in df.columns:
        # Create a temporary lowercase column for comparisons
        df['company_name_lower'] = df['company name'].astype(str).str.lower()

        # Define keyword lists
        education_keywords = ["univ", "college", "school", "university"]
        exclusion_keywords = [
            "llc", "inc.", "charity", "foundation", "hospital",
            "cancer center", "solutions", "universal", "insurance",
            "univar", "inc", "univers", "universe", "trust"
        ]

        # Vectorized check for education keywords using regex for whole word match
        education_condition = df['company_name_lower'].apply(lambda x: any(re.search(rf'\b{keyword}\b', x) for keyword in education_keywords))

        # Vectorized check for exclusion keywords using regex for whole word match
        exclusion_condition = df['company_name_lower'].apply(lambda x: any(re.search(rf'\b{keyword}\b', x) for keyword in exclusion_keywords))

        # Apply the logic to set the 'C' code only if it doesn't meet the exclusion conditions
        df.loc[(df['code'] == '') & education_condition & ~exclusion_condition, 'code'] = 'C'

        # Drop the temporary lowercase column
        df.drop(columns=['company_name_lower'], inplace=True)

    return df

def update_code_based_on_government(df):
    if "company name" in df.columns:
        # Create a temporary lowercase column for comparisons
        df['company_name_lower'] = df['company name'].apply(safe_str).str.lower()

        # Define keyword lists
        government_keywords = ["county", "city", "town", "state", "govt", "government"]
        exclusion_keywords = [
            "llc", "pllc", "inc.", "incorporated", "ltd", "company", "corp.", "corporation",
            "foundation", "charity", "non-profit", "inc", "medical", "works", "systems",
            "co.", "associates", "corp", "communications", "hospital", "hosp", "services",
            "solutions", "products", "biomedicine","controls"
        ]
        # Vectorized check for government keywords
        government_condition = df['company_name_lower'].apply(lambda x: any(re.search(rf'\b{keyword}\b', x) for keyword in government_keywords))

        # Vectorized check for exclusion keywords
        exclusion_condition = df['company_name_lower'].apply(lambda x: any(re.search(rf'\b{keyword}\b', x) for keyword in exclusion_keywords))

        # Apply the logic to set the 'G' code only if it doesn't meet the exclusion conditions
        df.loc[(df['code'] == '') & government_condition & ~exclusion_condition, 'code'] = 'G'

        # Remove the 'G' code if exclusion keywords are found
        df.loc[(df['code'] == 'G') & exclusion_condition, 'code'] = ''

        # Drop the temporary lowercase column
        df.drop(columns=['company_name_lower'], inplace=True)

    return df

def update_code_based_on_charity(df):
    def update_code(row):
        company_name = safe_str(row['company name']).lower()  # Use safe_str to ensure company name is a string
        if row['code'] == '':
            if any(keyword in company_name for keyword in ["charity", "foundation", "non-profit", "nonprofit"]) and not any(keyword in company_name for keyword in ["llc", "pllc", "inc.", "incorporated", "ltd", "company", "corp.", "corporation"]):
                return 'I'
        return row['code']

    if "company name" in df.columns:
        df["code"] = df.apply(update_code, axis=1)

    return df


def update_code_based_on_trust_estate(df):
    exclude_keywords = [
        "department", "llc", "pllc", "inc.", "incorporated", "ltd", "company",
        "corp.", "corporation", "foundation", "charity", "non-profit", "campaign",
        "county", "city", "town", "state", "govt", "government", "organization",
        "equipment", "north", "west", "east", "south", "service", "society",
        "comms.", "communications", "chevrolet", "audi", "nissan", "jeep",
        "dodge", "gmc", "cadillac", "ford", "honda", "hyundai", "group",
        "tesla", "dealership", "automotive", "toyota", "motors", "creative",
        "mitsubishi","association"
    ]

    def update_code(row):
        company_name = safe_str(row['company name']).lower()  # Use safe_str to ensure company name is a string
        if row['code'] == '':
            if "real estate" in company_name:
                return row['code']
            if re.search(r'\btrust\b', company_name) or "estate" in company_name:
                if not any(keyword in company_name for keyword in exclude_keywords):
                    return 'T'
        return row['code']

    if "company name" in df.columns:
        df["code"] = df.apply(update_code, axis=1)

    return df

def update_spendcountry(df):
    country_column = None
    if "country" in df.columns:
        country_column = "country"
    elif "address (country)" in df.columns:
        country_column = "address (country)"

    if country_column:
        df[country_column] = df[country_column].apply(clean_country_name)
        df["spendcountry"] = df[country_column].apply(normalize_country_name)
        print("SpendCountry values after normalization:")  # Debug: Print spendcountry values
        print(df["spendcountry"].head())  # Debug: Print the first few spendcountry values
    else:
        print("Country column not found in the dataframe.")  # Debug: Print if country column is missing
    return df

def update_po_box_column(df):
    def check_po_box(address):
        if isinstance(address, str) and any(term in address.lower() for term in ["po box", "p.o. box", "post office box"]):
            return 'YES'
        return ''

    if "complete full address" in df.columns:
        df["po box"] = df["complete full address"].apply(check_po_box)
    return df

def update_sanctioned_country(df, sanctioned_countries_list):
    if "spendcountry" in df.columns:
        print("Columns in sanctioned_countries_list:", sanctioned_countries_list.columns)  # Debug print
        print(sanctioned_countries_list.head())  # Debug print
        sanctioned_countries_set = set(sanctioned_countries_list['country'].str.lower())
        df['sanctioned country'] = df['spendcountry'].apply(lambda x: x if x.lower() in sanctioned_countries_set else '')
    return df

def reorder_columns(df):
    columns_order = ["code", "spendcountry", "sanctioned country", "internal supplier id", "company name", "complete full address"]

    # Add "email address", "contact email", "website domain", "tax id", and "PO Box" if they exist
    additional_columns = []
    if "contact email" in df.columns:
        additional_columns.append("contact email")
    if "website domain" in df.columns:
        additional_columns.append("website domain")
    if "email address" in df.columns:
        additional_columns.append("email address")
    if "tax id" in df.columns:
        additional_columns.append("tax id")
    if "po box" in df.columns:
        additional_columns.append("po box")
    additional_columns.append("unedited company name")

    columns_order.extend(additional_columns)

    remaining_columns = [col for col in df.columns if col not in columns_order]
    df = df[columns_order + remaining_columns]
    return df

def create_spend_summary_sheet(wb, spend_df):
    # Create a new sheet for Spend Summary
    spend_summary_sheet = wb.create_sheet(title="Spend Summary")

    # Calculate the aggregated spend for each unique internal supplier id
    spend_summary = spend_df.groupby('internal supplier id or vendor number*')['spend amount*'].sum().reset_index()
    spend_summary.columns = ['internal supplier id', 'aggregated spend']

    # Round the aggregated spend to two decimal places
    spend_summary['aggregated spend'] = spend_summary['aggregated spend'].round(2)

    # Calculate the total aggregated spend to compute percentages
    total_spend = spend_summary['aggregated spend'].sum()

    # Calculate spend percentage for each supplier
    spend_summary['spend percentage'] = spend_summary['aggregated spend'] / total_spend

    # Sort by aggregated spend in descending order
    spend_summary = spend_summary.sort_values(by='aggregated spend', ascending=False).reset_index(drop=True)

    # Write headers to the new sheet
    headers = ['internal supplier id', 'aggregated spend', 'spend percentage']
    spend_summary_sheet.append(headers)

    # Write the spend summary data to the sheet
    for idx, row in spend_summary.iterrows():
        spend_summary_sheet.append([row['internal supplier id'], row['aggregated spend'], row['spend percentage']])

    # Format percentage column
    for row in spend_summary_sheet.iter_rows(min_row=2, max_row=spend_summary_sheet.max_row, min_col=3, max_col=3):
        for cell in row:
            if cell.value != "":
                cell.number_format = numbers.FORMAT_PERCENTAGE_00

    # Set column widths for better readability
    spend_summary_sheet.column_dimensions['A'].width = 25
    spend_summary_sheet.column_dimensions['B'].width = 20
    spend_summary_sheet.column_dimensions['C'].width = 20

    # Apply teal fill and bold font to header
    for cell in spend_summary_sheet["1:1"]:
        cell.font = Font(bold=True)
        cell.fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")



def save_to_excel(df, output_file):
    df.to_excel(output_file, index=False)

def format_excel(output_file, df, col_mappings):
    wb = load_workbook(output_file)
    ws = wb.active
    ws.title = "Vendor Master"
    # Define the styles
    bold_font = Font(bold=True)
    teal_fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
    light_teal_fill = PatternFill(start_color="CCFFFF", end_color="CCFFFF", fill_type="solid")
    red_fill = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")
    light_red_fill = PatternFill(start_color="FFCCCC", end_color="FFCCCC", fill_type="solid")
    green_fill = PatternFill(start_color="00FF00", end_color="00FF00", fill_type="solid")
    light_green_fill = PatternFill(start_color="90EE90", end_color="90EE90", fill_type="solid")
    error_fill = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")

    # Apply header formatting
    teal_headers = ["internal supplier id", "code", "spendcountry", "sanctioned country", "company name", "complete full address"]
    red_headers = ["street address", "city", "state", "zip code", "country", "unedited company name"]
    green_headers = ["email address", "website domain", "contact email", "tax id", "po box"]

    for header in teal_headers:
        if header in df.columns:
            cell = ws.cell(row=1, column=df.columns.get_loc(header) + 1)
            cell.font = bold_font
            cell.fill = teal_fill

    for header in red_headers:
        if header in df.columns:
            cell = ws.cell(row=1, column=df.columns.get_loc(header) + 1)
            cell.font = bold_font
            cell.fill = red_fill

    for header in green_headers:
        if header in df.columns:
            cell = ws.cell(row=1, column=df.columns.get_loc(header) + 1)
            cell.font = bold_font
            cell.fill = green_fill

    # Apply cell formatting
    for row in tqdm(ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=len(df.columns)), desc="Formatting cells"):
        for cell in row:
            header = df.columns[cell.column - 1]
            if header in teal_headers:
                cell.fill = light_teal_fill
            elif header in red_headers:
                cell.fill = light_red_fill
            elif header in green_headers:
                cell.fill = light_green_fill
            # Highlight cells with "ERROR" in spendcountry column
            if header == "spendcountry" and cell.value == "ERROR":
                cell.fill = error_fill
    # Set column widths
    ws.column_dimensions['B'].width = 15  # SPENDCOUNTRY
    ws.column_dimensions['C'].width = 15  # SANCTIONED COUNTRY
    ws.column_dimensions['D'].width = 20  # INTERNAL SUPPLIER ID
    ws.column_dimensions['E'].width = 25  # COMPANY NAME
    ws.column_dimensions['F'].width = 50  # COMPLETE FULL ADDRESS
    if "email address" in df.columns:
        ws.column_dimensions['G'].width = 25  # EMAIL ADDRESS
    if "contact email" in df.columns:
        ws.column_dimensions['H'].width = 25  # CONTACT EMAIL
    if "website domain" in df.columns:
        ws.column_dimensions['I'].width = 25  # WEBSITE DOMAIN
    if "tax id" in df.columns:
        ws.column_dimensions['J'].width = 25  # TAX ID
    if "po box" in df.columns:
        ws.column_dimensions['K'].width = 25  # PO BOX
    ws.column_dimensions['L'].width = 25  # UNEDITED COMPANY NAME

    # Apply header formatting for remaining columns
    remaining_columns = [col for col in df.columns if col not in teal_headers + red_headers + green_headers]
    for col in remaining_columns:
        cell = ws.cell(row=1, column=df.columns.get_loc(col) + 1)
        cell.font = bold_font
        cell.fill = red_fill  # Use red for headers of remaining columns

    # Create the Summary sheet
    code_summary_sheet = wb.create_sheet(title="Code Summary")

    # Calculate unique counts of `internal supplier id` for each code
    unique_counts = df.groupby("code")["internal supplier id"].nunique().reindex(["A", "P", "G", "C", "I", "T"], fill_value=0)

    # GRAND TOTAL: Total unique `internal supplier id`
    grand_total = df["internal supplier id"].nunique()

    # Create summary data
    summary_data = {
        "Label": ["A", "P", "G", "C", "I", "T", "GRAND TOTAL"],
        "Type": ["Client Employee", "Person", "Govt Entity", "College", "Non-Profit", "Trust/Estate", ""],
        "Count": list(unique_counts) + [grand_total],
        "Percentage": list(unique_counts / grand_total) + [""]
    }

    summary_df = pd.DataFrame(summary_data)

    # Write headers and data to the sheet
    for r in dataframe_to_rows(summary_df, index=False, header=True):
        code_summary_sheet.append(r)

    # Apply formatting to Summary sheet
    for cell in code_summary_sheet["1:1"]:
        cell.font = bold_font
        cell.fill = teal_fill  # Apply teal fill to header

    # Set column widths for Summary sheet
    code_summary_sheet.column_dimensions['A'].width = 15
    code_summary_sheet.column_dimensions['B'].width = 15

    # Apply light teal fill to the last row
    for cell in code_summary_sheet[f"A{code_summary_sheet.max_row}":f"D{code_summary_sheet.max_row}"][0]:
        cell.fill = light_teal_fill

    # Format the percentage column
    for row in code_summary_sheet.iter_rows(min_row=2, max_row=code_summary_sheet.max_row, min_col=4, max_col=4):
        for cell in row:
            if cell.value != "":
                cell.number_format = numbers.FORMAT_PERCENTAGE_00

    # Create the SpendCountry sheet
    spendcountry_counts = df.drop_duplicates(subset=["internal supplier id", "spendcountry"]).replace('', '(blank)').value_counts("spendcountry").reset_index()
    spendcountry_counts.columns = ["SpendCountry", "SpendCountry Count"]
    spendcountry_counts["SpendCountry Percentage"] = spendcountry_counts["SpendCountry Count"] / grand_total
    sc_grand_total_row = pd.DataFrame({"SpendCountry": ["SC GRAND TOTAL"], "SpendCountry Count": [grand_total], "SpendCountry Percentage": [""]})
    spendcountry_counts = pd.concat([spendcountry_counts, sc_grand_total_row], ignore_index=True)

    spendcountry_sheet = wb.create_sheet(title="SpendCountry")
    spendcountry_headers = ["SpendCountry", "SpendCountry Count", "SpendCountry Percentage"]
    spendcountry_sheet.append(spendcountry_headers)

    for idx, row in spendcountry_counts.iterrows():
        spendcountry_sheet.append([row["SpendCountry"], row["SpendCountry Count"], row["SpendCountry Percentage"]])

    # Format percentage columns in SpendCountry sheet
    for row in spendcountry_sheet.iter_rows(min_row=2, max_row=spendcountry_sheet.max_row, min_col=3, max_col=3):
        for cell in row:
            if cell.value != "":
                cell.number_format = numbers.FORMAT_PERCENTAGE_00

    # Apply formatting to SpendCountry sheet headers
    for cell in spendcountry_sheet["1:1"]:
        cell.font = bold_font
        cell.fill = teal_fill  # Apply teal fill to header

    # Set column widths for SpendCountry sheet
    for col in spendcountry_sheet.columns:
        spendcountry_sheet.column_dimensions[col[0].column_letter].width = 25

    # Apply two-color scale to SpendCountry column C
    spendcountry_sheet = wb["SpendCountry"]
    color_scale_rule = ColorScaleRule(
        start_type="min", start_color=Color("FFFFFF"),  # No color for minimum
        end_type="max", end_color=Color("FFA500")  # Orange color for maximum
    )

    # Apply light teal fill to the last row
    for cell in spendcountry_sheet["A{}".format(spendcountry_sheet.max_row):"C{}".format(spendcountry_sheet.max_row)][0]:
        cell.fill = light_teal_fill

    spendcountry_sheet.conditional_formatting.add("C2:C{}".format(spendcountry_sheet.max_row), color_scale_rule)

    # Create the SpendState sheet
    spendstate_counts = df[df['spendcountry'] == 'United States'].drop_duplicates(subset=["internal supplier id", "address (state/province)"])
    spendstate_counts = spendstate_counts.value_counts("address (state/province)").reset_index()
    spendstate_counts.columns = ["SpendState", "SpendState Count"]
    spendstate_counts["SpendState Percentage"] = spendstate_counts["SpendState Count"] / spendstate_counts["SpendState Count"].sum()
    ss_grand_total_row = pd.DataFrame({"SpendState": ["SS GRAND TOTAL"], "SpendState Count": [spendstate_counts["SpendState Count"].sum()], "SpendState Percentage": [""]})
    spendstate_counts = pd.concat([spendstate_counts, ss_grand_total_row], ignore_index=True)

    spendstate_sheet = wb.create_sheet(title="SpendState")
    spendstate_headers = ["SpendState", "SpendState Count", "SpendState Percentage"]
    spendstate_sheet.append(spendstate_headers)

    for idx, row in spendstate_counts.iterrows():
        spendstate_sheet.append([row["SpendState"], row["SpendState Count"], row["SpendState Percentage"]])

    # Format percentage columns in SpendState sheet
    for row in spendstate_sheet.iter_rows(min_row=2, max_row=spendstate_sheet.max_row, min_col=3, max_col=3):
        for cell in row:
            if cell.value != "":
                cell.number_format = numbers.FORMAT_PERCENTAGE_00

    # Apply formatting to SpendState sheet headers
    for cell in spendstate_sheet["1:1"]:
        cell.font = bold_font
        cell.fill = teal_fill  # Apply teal fill to header

    # Set column widths for SpendState sheet
    for col in spendstate_sheet.columns:
        spendstate_sheet.column_dimensions[col[0].column_letter].width = 25

    # Apply two-color scale to SpendState column C
    spendstate_sheet = wb["SpendState"]
    spendstate_sheet.conditional_formatting.add("C2:C{}".format(spendstate_sheet.max_row), color_scale_rule)

    # Apply light teal fill to the last row
    for cell in spendstate_sheet["A{}".format(spendstate_sheet.max_row):"C{}".format(spendstate_sheet.max_row)][0]:
        cell.fill = light_teal_fill

    # Create the Address Gaps sheet
    address_gaps_sheet = wb.create_sheet(title="Address Gaps")
    address_gaps_headers = ["Number of TOTAL UNIQUE ID Addresses", "# of Unique Addresses With Gaps", "Street Address Gaps", "City Gaps", "State Gaps", "Country Gaps", "Zip Code Gaps"]
    address_gaps_sheet.append(address_gaps_headers)

    # Calculate the values for the Address Gaps sheet
    total_unique_ids = df["internal supplier id"].nunique()

    # Filter out empty columns in col_mappings
    non_empty_cols = {k: v for k, v in col_mappings.items() if v}

    # Calculate gaps for unique addresses (relative to unique "internal supplier id")
    unique_id_groups = df.groupby("internal supplier id")

    # Initialize counters for gaps in each address column
    street_address_gaps = 0
    city_gaps = 0
    state_gaps = 0
    country_gaps = 0
    zip_code_gaps = 0

    # Check gaps within each group for each address component
    for _, group in unique_id_groups:
        # Check if any row in each group has a gap in the specified column
        if non_empty_cols.get("street address") and group[non_empty_cols["street address"]].apply(lambda x: str(x).strip() == '').any():
            street_address_gaps += 1
        if non_empty_cols.get("city") and group[non_empty_cols["city"]].apply(lambda x: str(x).strip() == '').any():
            city_gaps += 1
        if non_empty_cols.get("state") and group[non_empty_cols["state"]].apply(lambda x: str(x).strip() == '').any():
            state_gaps += 1
        if non_empty_cols.get("country") and group[non_empty_cols["country"]].apply(lambda x: str(x).strip() == '').any():
            country_gaps += 1
        if non_empty_cols.get("zip code") and group[non_empty_cols["zip code"]].apply(lambda x: str(x).strip() == '').any():
            zip_code_gaps += 1

    # Count unique addresses with any gap
    unique_addresses_with_gaps = sum(
        unique_id_groups.apply(lambda x: any(x[non_empty_cols[col]].apply(lambda y: str(y).strip() == '').any() for col in non_empty_cols))
    )

    # Prepare and append the address gaps data to the sheet
    address_gaps_data = [total_unique_ids, unique_addresses_with_gaps, street_address_gaps, city_gaps, state_gaps, country_gaps, zip_code_gaps]
    address_gaps_sheet.append(address_gaps_data)

    # Calculate and set static percentage value in B3
    percentage_value = unique_addresses_with_gaps / total_unique_ids if total_unique_ids != 0 else 0
    address_gaps_sheet["B3"].value = percentage_value
    address_gaps_sheet["B3"].number_format = numbers.FORMAT_PERCENTAGE_00

    # Apply formatting to Address Gaps sheet headers
    for cell in address_gaps_sheet["1:1"]:
        cell.font = bold_font
        cell.fill = teal_fill  # Apply teal fill to header

    # Set column widths for Address Gaps sheet
    for col in address_gaps_sheet.columns:
        address_gaps_sheet.column_dimensions[col[0].column_letter].width = 25

    # Apply light teal fill to cell B3 in Address Gaps
    address_gaps_sheet = wb["Address Gaps"]
    address_gaps_sheet["B3"].fill = light_teal_fill

    # Save the workbook with the applied styles
    wb.save(output_file)

def create_vm_vs_spend_id_check_sheet(wb, vm_df, spend_df):
    vs_spend_sheet = wb.create_sheet(title="VM vs. Spend ID Check")
    headers = ["VM Unique IDs", "VM ID in Spend tab?", "Spend Unique IDs", "Spend ID in VM tab?"]
    vs_spend_sheet.append(["VM Unique Missing", "", "Spend Unique Missing", ""])
    vs_spend_sheet.append(headers)

    vm_unique_ids = vm_df["internal supplier id or vendor number"].drop_duplicates()
    spend_unique_ids = spend_df["internal supplier id or vendor number*"].drop_duplicates()

    vm_data = []
    for vm_id in vm_unique_ids:
        vm_in_spend = "YES" if vm_id in spend_unique_ids.values else "NO"
        vm_data.append((vm_id, vm_in_spend))

    spend_data = []
    for spend_id in spend_unique_ids:
        spend_in_vm = "YES" if spend_id in vm_unique_ids.values else "NO"
        spend_data.append((spend_id, spend_in_vm))

    max_rows = max(len(vm_data), len(spend_data))
    for i in range(max_rows):
        vm_id, vm_in_spend = vm_data[i] if i < len(vm_data) else ("", "")
        spend_id, spend_in_vm = spend_data[i] if i < len(spend_data) else ("", "")
        vs_spend_sheet.append([vm_id, vm_in_spend, spend_id, spend_in_vm])

    vm_missing_count = len([cell for cell in vs_spend_sheet['B'][2:] if cell.value == "NO"])
    spend_missing_count = len([cell for cell in vs_spend_sheet['D'][2:] if cell.value == "NO"])

    vs_spend_sheet['B1'].value = vm_missing_count
    vs_spend_sheet['D1'].value = spend_missing_count

    thin_border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
    for cell in ["A1", "B1"]:
        vs_spend_sheet[cell].fill = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")
        vs_spend_sheet[cell].border = thin_border
        vs_spend_sheet[cell].font = Font(bold=True)
    for cell in ["C1", "D1"]:
        vs_spend_sheet[cell].fill = PatternFill(start_color="90EE90", end_color="90EE90", fill_type="solid")
        vs_spend_sheet[cell].border = thin_border
        vs_spend_sheet[cell].font = Font(bold=True)

    # Apply teal highlight and bold font to header row (A2:D2)
    for cell in ["A2", "B2", "C2", "D2"]:
        vs_spend_sheet[cell].fill = PatternFill(start_color="00FFFF", end_color="00FFFF", fill_type="solid")
        vs_spend_sheet[cell].font = Font(bold=True)

    for cell in vs_spend_sheet['B'][2:]:
        if cell.value == "NO":
            cell.fill = PatternFill(start_color="FF6666", end_color="FF6666", fill_type="solid")
    for cell in vs_spend_sheet['D'][2:]:
        if cell.value == "NO":
            cell.fill = PatternFill(start_color="FF6666", end_color="FF6666", fill_type="solid")

    for col in ["A", "B", "C", "D"]:
        vs_spend_sheet.column_dimensions[col].width = 25


def main(file_path, sanctioned_countries_file):
    start_time = time.time()  # Start time
    df = read_excel_file(file_path, sheet_name="Vendor Master")
    sanctioned_countries_list = read_excel_file(sanctioned_countries_file, sheet_name=None)  # Read entire file

    # Check required columns early
    col_mappings = check_required_columns(df)
    print("Column mappings:", col_mappings)  # Debug: Print column mappings

    # Enforce validation: ensure required columns are present
    missing_columns = [key for key, value in col_mappings.items() if value == '']
    if missing_columns:
        raise ValueError(f"Missing required columns in the input file: {', '.join(missing_columns)}")

    row_count = df.shape[0]  # Number of rows
    df = add_columns(df)  # Ensure columns are added

    # Generate the "Addr. Gaps to Start" data before any address cleaning
    output_gaps_file = create_addr_gaps_start_workbook(file_path, df)

#    col_mappings = check_required_columns(df)
#    print("Column mappings:", col_mappings)  # Debug information

#    if not any(col_mappings.values()):
#        if 'complete full address' in df.columns:
#            df = df.rename(columns={'complete full address': 'complete full address'})
#        else:
#            print("Required address columns or 'Complete Full Address' column is missing. Exiting script.")
#            return

    # Format US ZIP codes to add a leading zero if necessary
    df = format_us_zip_codes(df)

    # Fill missing city and state based on ZIP code
    df = fill_missing_city_state(df)
    df = create_or_clean_complete_address(df, col_mappings)
    df = clean_columns(df)

    # Process 'P' inputs using Hugging Face and common names libraries
    nasdaq_100_companies = get_nasdaq_100_companies()
    company_names_person_names = get_company_names_person_names()
    combined_names = set(nasdaq_100_companies + company_names_person_names)
    common_names1 = get_common_names_from_names_lib()
    df = normalize_state_province(df)
    df = update_code_based_on_ner(df, common_names1, company_names_person_names)
    df = update_code_based_on_common_names(df, common_names1)
    df = remove_p_based_on_combined_names(df, combined_names)
    df = update_country_based_on_city_state(df)
    df['spendcountry'] = df['spendcountry'].apply(normalize_country_name)

    # Update 'A' codes based on contact email and client domain
    df = update_code_based_on_contact_email(df, client_domain)
    df = update_code_based_on_company_name(df)
    df = update_code_based_on_government(df)
    df = update_code_based_on_charity(df)
    df = update_code_based_on_trust_estate(df)
    df = update_spendcountry(df)
    df = update_sanctioned_country(df, sanctioned_countries_list)
    df = update_po_box_column(df)
    # Ensure all necessary columns are present before reordering
    necessary_columns = ["spendcountry", "sanctioned country", "internal supplier id"]
    for col in necessary_columns:
        if col not in df.columns:
            df[col] = ''

    df = reorder_columns(df)

    # Generate a unique filename with today's date
    today = datetime.today().strftime('%Y-%m-%d')
    output_file = f"Step 1_{client_name} - SDP - VM Analysis - {today}.xlsx"

    # Save initial data to the main output file
    save_to_excel(df, output_file)

    # Format the main output file
    format_excel(output_file, df, col_mappings)

    # Add the "Addr. Gaps to Start" data to the main workbook
    copy_addr_gaps_data_to_main(output_file, output_gaps_file)

    # Run initial address checks before processing for "Cleaning Summary" tab
    add_address_gaps_and_cleaning_summary(output_file, df)

    # Load the Vendor Master and Spend Data tabs for the VM vs. Spend ID Check
    vm_df = read_excel_file(file_path, sheet_name="Vendor Master")
    spend_df = read_excel_file(file_path, sheet_name="Spend Data")

    # Create the VM vs. Spend ID Check sheet
    wb = load_workbook(output_file)
    create_vm_vs_spend_id_check_sheet(wb, vm_df, spend_df)

    # Create the Spend Summary sheet
    create_spend_summary_sheet(wb, spend_df)

    # Save the final workbook with the new tab
    wb.save(output_file)

    # Calculate elapsed time
    end_time = time.time()  # End time
    elapsed_time = end_time - start_time

    # Print the elapsed time and row count
    print(f"Script execution time: {elapsed_time:.2f} seconds")
    print(f"Number of rows processed: {row_count}")

    # Make sure the file appears in the file system for download
    from google.colab import files
    files.download(output_file)

# Run the main function
main(input_file, sanctioned_countries_file)

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


What is the client name? affirm
What is the client domain? affirm.com
Columns in input file: ['internal supplier id or vendor number', 'company name', 'website domain', 'merchant_platform', 'is_shopify_merchant', 'address (street name and number)', 'address (city)', 'address (state/province)', 'address (zip code / postal code)', 'address (country)']
Columns in input file: ['country']
Column mappings: {'street address': 'address (street name and number)', 'city': 'address (city)', 'state': 'address (state/province)', 'country': 'address (country)', 'zip code': 'address (zip code / postal code)'}
Column mappings: {'street address': 'address (street name and number)', 'city': 'address (city)', 'state': 'address (state/province)', 'country': 'address (country)', 'zip code': 'address (zip code / postal code)'}


<ipython-input-8-156cc570a4cc>:84: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rows_with_any_gap = unique_id_groups.apply(


Complete full address updated.
0       13721 sw 84 th st Unit G, Miami, FL, 33183, US
1        124 West 4th Street, Royal Oak, MI, 48067, US
2       7 Ridge Run Southeast, Marietta, GA, 30067, US
3                                                     
4    3203 West McKinley Blvd, Milwaukee, WI, 53234, US
Name: complete full address, dtype: object


ERROR:yfinance:404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/ATVI?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=ATVI&crumb=DAej8fTcxSc
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
  0%|          | 20/45000 [00:0

...

 48%|████▊     | 21415/45000 [1:03:25<46:13,  8.50it/s]  

Assigning 'P' to company name: sweet september lane


 48%|████▊     | 21419/45000 [1:03:26<1:09:31,  5.65it/s]

Assigning 'P' to company name: rashida's creations


 48%|████▊     | 21476/45000 [1:03:36<1:38:58,  3.96it/s]